In [1]:
import numpy as np
import pyvista as pv
import gdist
from scipy.spatial.distance import cdist

from phd_helpers.paths import get_boundary

# Identify surface bone patches for RP coupling for BCs

In [2]:
mesh = pv.read('../volumetricMesh/cgal-box/tpm-final-postioned.vtu')

In [32]:
def bone_surface_patch_nodes(mesh, patch_dist, distance_measure="euclidean", only_full_face_nodes=True):
    """
    mesh: FE ready (positioned with region_id)\n
    patch_dist: min distance of nodes from cartilage edge e.g. float/int ; or within given x coordinates e.g. [-100, -5] if xlims\n
    distance_measure: "geodesic" or "euclidean" or "xlims"\n
    only_full_face_nodes: if True, remove nodes that do not form part of a complete mesh surface face\n\n
    Returns the node ids on the given mesh 
    """
    mesh['node_id'] = np.arange(mesh.n_points)
    mesh['cell_id'] = np.arange(mesh.n_cells)
    bone_surf = mesh.extract_cells(np.where(mesh['region_id']==-1)[0]).extract_geometry()
    bone_boudnary = get_boundary(bone_surf)
    boundary_mask = np.isin(bone_surf['node_id'], bone_boudnary['node_id']) # on bone surf

    if distance_measure == "geodesic":
        # min distance for every point from boundary points (inc. boundary points)
        dists = gdist.compute_gdist( 
            bone_surf.points.astype(np.float64),
            bone_surf.faces.reshape(-1, 4)[:, 1:].astype(np.int32),
            source_indices = np.arange(bone_surf.n_points)[boundary_mask].astype(np.int32)
        )
        bone_patch_mask = dists >= patch_dist # mask of nodes on bone surf 

    elif distance_measure == "euclidean":
        # min distance for every point from boundary points (inc. boundary points)
        dists = cdist(bone_surf.points[boundary_mask],  bone_surf.points).min(axis=0)
        bone_patch_mask = dists >= patch_dist # mask of nodes on bone surf 

    elif distance_measure == "xlims":
        Xs = bone_surf.points[:, 0]
        bone_patch_mask = (Xs>=patch_dist[0]) & (Xs<=patch_dist[1])

    else:
        raise ValueError("Invalid distance measure. Choose from: 'geodesic' or 'euclidean'")


    if only_full_face_nodes:
        extracted_surf = bone_surf.extract_points(bone_patch_mask, adjacent_cells=False).extract_geometry()
        bone_patch_mask = np.isin(bone_surf['node_id'], extracted_surf['node_id'])
        mesh['bc_patch'] = np.zeros(mesh.n_cells)
        mesh.cell_data['bc_patch'][extracted_surf['cell_id']] = 1

    # output node ids
    bone_patch_nodes = bone_surf['node_id'][bone_patch_mask] # nodes on full mesh

    # label output nodes on original mesh
    mesh['bc_patch'] = np.zeros(mesh.n_points)
    mesh.point_data['bc_patch'][bone_patch_nodes] = 1

    return bone_patch_nodes

In [33]:
bone_set_dist = [-100, 0] # distance from cartilage edge (or min and max x_coords)
distance_measure = "xlims" # "geodesic" or "euclidean" or "x_lims"
only_full_face_nodes = True # remove nodes that do not form part of a complete mesh surface face
bone_set_nodes = bone_surface_patch_nodes(mesh, bone_set_dist, distance_measure, only_full_face_nodes)

In [35]:
pl = pv.Plotter()
surf_mesh = mesh.extract_cells_by_type(5)
pl.add_mesh(surf_mesh, scalars=surf_mesh.cell_data['bc_patch'])
pl.show()

Widget(value='<iframe src="http://localhost:52585/index.html?ui=P_0x32bce29c0_16&reconnect=auto" class="pyvist…

# Convert tet4 to tet10

In [3]:
from phd_helpers.paths import linear_to_quadratic_mesh, quadratic_to_linear_mesh

In [4]:
bone = 'tpm'
tpm = pv.read(f'../volumetricMesh/cgal-box/tpm-final-postioned.vtu')
mc1 = pv.read(f'../volumetricMesh/cgal-box/mc1-final-postioned.vtu')

tpm10 = linear_to_quadratic_mesh(tpm)
mc110 = linear_to_quadratic_mesh(mc1)

lin = quadratic_to_linear_mesh(tpm10)

print(tpm.n_points)
print(tpm10.n_points)
print(lin.n_points)


#tpm10.save(f'../volumetricMesh/cgal-box/tpm-final-postioned-quad.vtu')
#mc110.save(f'../volumetricMesh/cgal-box/mc1-final-postioned-quad.vtu')

45109
344610
45109


In [88]:
tpm10.linear_copy().extract_cells_by_type(5).plot(show_edges=True, scalars='region_id')
mc110.linear_copy().extract_cells_by_type(5).plot(show_edges=True, scalars='region_id')

Widget(value='<iframe src="http://localhost:50545/index.html?ui=P_0x30ef62660_28&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:50545/index.html?ui=P_0x308012090_29&reconnect=auto" class="pyvist…

### test bone_patch_nodes on quad mesh

In [89]:
from AbaqusPreprocessing import bone_surface_patch_nodes

In [94]:
bone_set_dist = [-100, 0] # distance from cartilage edge (or min and max x_coords)
distance_measure = "x_lims" # "geodesic" or "euclidean" or "x_lims"
only_full_face_nodes = True # remove nodes that do not form part of a complete mesh surface face
bone_set_nodes = bone_surface_patch_nodes(tpm10, bone_set_dist, distance_measure, only_full_face_nodes)

In [96]:
pl = pv.Plotter()
pl.add_mesh(tpm10, color='white')
pl.add_points(tpm10.points[bone_set_nodes])
pl.show()

Widget(value='<iframe src="http://localhost:50545/index.html?ui=P_0x131cbfe30_34&reconnect=auto" class="pyvist…